# Solución Lab 07: Cálculo de KPIs y Generación Silver/Gold
## TransCore Data Engineer - Módulo 07

## 1. Overview y Configuración

Este notebook implementa la solución completa del **Laboratorio 07** donde calcularemos:
- **KPIs de disponibilidad**: Disponibilidad %, MTBF, MTTR
- **Capa Silver**: Dataset enriquecido con metadatos de activos
- **Capa Gold**: Agregaciones de negocio por tipo y ubicación

**Case Study**: TransCore - Sistema de gestión de activos de obra lineal

In [ ]:
# Configuración de credenciales AWS (S3 real)
import os

os.environ["AWS_ACCESS_KEY_ID"] = "<TU_ACCESS_KEY_ID>"
os.environ["AWS_SECRET_ACCESS_KEY"] = "<TU_SECRET_ACCESS_KEY>"

# Limpiar cualquier configuración de LocalStack
os.environ.pop("AWS_ENDPOINT_URL", None)
os.environ.pop("AWS_ENDPOINT", None)

# Configurar región de AWS explícitamente para AWS real
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

print("Configuración: AWS real, región eu-west-1")

## 2. Importar Libraries

In [ ]:
import numpy as np
import pandas as pd
import boto3
from datetime import datetime, timedelta
from pathlib import Path

print("Librerías importadas correctamente")

## 3. Configuración de S3 y Buckets

In [ ]:
# Configuración S3 - Usando el bucket creado en Lab 06
BUCKET_NAME = "<TU_BUCKET_NAME>"

# Prefijos de las capas medallón
SILVER_PREFIX = "silver"
GOLD_PREFIX = "gold"

# Crear cliente S3
s3_client = boto3.client("s3", region_name="eu-west-1")

print(f"Bucket: {BUCKET_NAME}")
print(f"Silver prefix: {SILVER_PREFIX}")
print(f"Gold prefix: {GOLD_PREFIX}")

## 4. Paso 1: Cargar y Explorar los Datos

In [ ]:
# Los datos están en el bucket S3 en la capa landing
# Cargamos directamente desde S3 usando boto3

def load_csv_from_s3(bucket, key):
    """Carga un CSV directamente desde S3 a un DataFrame."""
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(obj['Body'])

def load_parquet_from_s3(bucket, key):
    """Carga un Parquet directamente desde S3 a un DataFrame."""
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(obj['Body'])

# Cargar datasets desde S3
print("Cargando datasets desde S3...")

# Telemetria - puede estar particionada por fecha
telemetria = load_csv_from_s3(BUCKET_NAME, "landing/telemetria_parque_mes.csv")
print(f"✓ Telemetria: {len(telemetria):,} registros")

# Activos en formato Parquet
activos = load_parquet_from_s3(BUCKET_NAME, "raw/activos.parquet")
print(f"✓ Activos: {len(activos):,} registros")

# Órdenes de mantenimiento
ordenes = load_csv_from_s3(BUCKET_NAME, "landing/ordenes_mantenimiento.csv")
print(f"✓ Órdenes mantenimiento: {len(ordenes):,} registros")

In [ ]:
# Exploración inicial de telemetria
print("=== Telemetria ===")
print(f"Forma: {telemetria.shape}")
print(f"\nColumnas: {telemetria.columns.tolist()}")
print(f"\nTipos de datos:\n{telemetria.dtypes}")
print(f"\nPrimeras 5 filas:\n{telemetria.head()}")

In [ ]:
# Exploración de activos
print("=== Activos ===")
print(f"Forma: {activos.shape}")
print(f"\nColumnas: {activos.columns.tolist()}")
print(f"\nTipos de datos:\n{activos.dtypes}")
print(f"\nPrimeras 5 filas:\n{activos.head()}")

In [ ]:
# Exploración de órdenes de mantenimiento
print("=== Órdenes de Mantenimiento ===")
print(f"Forma: {ordenes.shape}")
print(f"\nColumnas: {ordenes.columns.tolist()}")
print(f"\nPrimeras 5 filas:\n{ordenes.head()}")

## 5. Paso 2: Limpieza de Datos

### 2.1 Convertir timestamp a datetime

In [ ]:
# Convertir timestamp a datetime
telemetria['timestamp'] = pd.to_datetime(telemetria['timestamp'])
print(f"Timestamp convertido a datetime: {telemetria['timestamp'].dtype}")

# Verificar rango de fechas
print(f"\nRango de fechas: {telemetria['timestamp'].min()} a {telemetria['timestamp'].max()}")

### 2.2 Tratar valores nulos

In [ ]:
# Verificar valores nulos antes de limpieza
print("=== Valores nulos antes de limpieza ===")
print(telemetria.isnull().sum())
print(f"\nTotal registros antes: {len(telemetria):,}")

In [ ]:
# Tratar nulos en temperatura y vibracion (usar mediana)
telemetria['temperatura'] = telemetria['temperatura'].fillna(telemetria['temperatura'].median())
telemetria['vibracion'] = telemetria['vibracion'].fillna(telemetria['vibracion'].median())

# Tratar nulos en estado_operativo (forward fill)
telemetria['estado_operativo'] = telemetria['estado_operativo'].fillna(method='ffill')

print("=== Valores nulos después de imputación ===")
print(telemetria.isnull().sum())

### 2.3 Eliminar duplicados

In [ ]:
# Eliminar duplicados
registros_antes = len(telemetria)
telemetria = telemetria.drop_duplicates()
print(f"Duplicados eliminados: {registros_antes - len(telemetria):,}")
print(f"Total después: {len(telemetria):,}")

### 2.4 Filtrar outliers de vibración (3 desviaciones estándar)

In [ ]:
# Filtrar outliers de vibración (超出3个标准差)
vibracion_mean = telemetria['vibracion'].mean()
vibracion_std = telemetria['vibracion'].std()

limite_inferior = vibracion_mean - 3 * vibracion_std
limite_superior = vibracion_mean + 3 * vibracion_std

print(f"Media vibración: {vibracion_mean:.2f}")
print(f"Std vibración: {vibracion_std:.2f}")
print(f"Rango válido: [{limite_inferior:.2f}, {limite_superior:.2f}]")

registros_antes = len(telemetria)
telemetria = telemetria[
    (telemetria['vibracion'] >= limite_inferior) & 
    (telemetria['vibracion'] <= limite_superior)
]
print(f"\nOutliers eliminados: {registros_antes - len(telemetria):,}")
print(f"Total después: {len(telemetria):,}")

## 6. Paso 3: Calcular Tiempo Operativo por Activo

In [ ]:
# Ordenar datos por activo_id y timestamp
telemetria = telemetria.sort_values(['activo_id', 'timestamp']).reset_index(drop=True)

# Calcular duración entre lecturas (en minutos)
telemetria['timestamp_prev'] = telemetria.groupby('activo_id')['timestamp'].shift(1)
telemetria['duracion_min'] = (telemetria['timestamp'] - telemetria['timestamp_prev']).dt.total_seconds() / 60

# Para la primera lectura de cada activo, asumimos duración típica de 5 minutos
telemetria['duracion_min'] = telemetria['duracion_min'].fillna(5)

# Para la última lectura de cada activo, también asumimos 5 minutos
ultima_lectura = telemetria.groupby('activo_id')['timestamp'].transform('max')
telemetria.loc[telemetria['timestamp'] == ultima_lectura, 'duracion_min'] = 5

# Calcular tiempo operativo
telemetria['tiempo_operativo_min'] = telemetria['duracion_min'] * telemetria['estado_operativo']

print("=== Muestra de cálculo de tiempo ===")
print(telemetria[['activo_id', 'timestamp', 'estado_operativo', 'duracion_min', 'tiempo_operativo_min']].head(10))

In [ ]:
# Agregar por activo
resumen_tiempo = telemetria.groupby('activo_id').agg(
    tiempo_total_min=('duracion_min', 'sum'),
    tiempo_operativo_min=('tiempo_operativo_min', 'sum')
).reset_index()

print("=== Resumen de Tiempo por Activo (primeros 10) ===")
print(resumen_tiempo.head(10))
print(f"\nTotal activos: {len(resumen_tiempo)}")

## 7. Paso 4: Detectar Fallos y Calcular Tiempos de Reparación

In [ ]:
# Detectar transiciones de estado
# Fallo: estado pasa de 1 a 0
# Reparación: estado pasa de 0 a 1

telemetria = telemetria.sort_values(['activo_id', 'timestamp']).reset_index(drop=True)
telemetria['estado_prev'] = telemetria.groupby('activo_id')['estado_operativo'].shift(1)

# Identificar fallos y reparaciones
telemetria['es_fallo'] = (telemetria['estado_prev'] == 1) & (telemetria['estado_operativo'] == 0)
telemetria['es_reparacion'] = (telemetria['estado_prev'] == 0) & (telemetria['estado_operativo'] == 1)

print(f"Total fallos detectados: {telemetria['es_fallo'].sum()}")
print(f"Total reparaciones detectadas: {telemetria['es_reparacion'].sum()}")

In [ ]:
# Para cada fallo, calcular tiempo hasta la siguiente reparación
# Primero, marcar cada periodo de inactividad
telemetria['periodo_inactividad'] = (telemetria['estado_operativo'] == 0).cumsum()

# Calcular duración de cada periodo de inactividad
def calcular_tiempo_reparacion(group):
    """Calcula el tiempo de reparación para un activo."""
    if len(group) < 2:
        return pd.Series({'num_fallos': 0, 'num_reparaciones': 0, 'tiempo_reparacion_min': 0})
    
    fallos = group['es_fallo'].sum()
    reparaciones = group['es_reparacion'].sum()
    
    # Tiempo entre primer fallo y última reparación
    tiempos_reparacion = []
    for i, row in group.iterrows():
        if row['es_fallo']:
            # Buscar la siguiente reparación
            idx = group.index.get_loc(i)
            for j in range(idx + 1, len(group)):
                if group.iloc[j]['es_reparacion']:
                    tiempo_reparacion = (group.iloc[j]['timestamp'] - row['timestamp']).total_seconds() / 60
                    tiempos_reparacion.append(tiempo_reparacion)
                    break
    
    tiempo_total_reparacion = sum(tiempos_reparacion) if tiempos_reparacion else 0
    
    return pd.Series({
        'num_fallos': fallos,
        'num_reparaciones': reparaciones,
        'tiempo_reparacion_min': tiempo_total_reparacion
    })

# Aplicar por cada activo
fallos = telemetria.groupby('activo_id').apply(calcular_tiempo_reparacion).reset_index()

print("=== Resumen de Fallos por Activo (primeros 10) ===")
print(fallos.head(10))

## 8. Paso 5: Calcular KPIs Finales

In [ ]:
# Combinar resumen de tiempo con fallos
kpis = resumen_tiempo.merge(fallos, on='activo_id', how='left')

# Calcular KPIs evitando división por cero
kpis['disponibilidad_pct'] = np.where(
    kpis['tiempo_total_min'] > 0,
    (kpis['tiempo_operativo_min'] / kpis['tiempo_total_min']) * 100,
    0
)

kpis['mtbf_min'] = np.where(
    kpis['num_fallos'] > 0,
    kpis['tiempo_operativo_min'] / kpis['num_fallos'],
    kpis['tiempo_operativo_min']  # Si no hay fallos, MTBF = tiempo operativo total
)

kpis['mttr_min'] = np.where(
    kpis['num_reparaciones'] > 0,
    kpis['tiempo_reparacion_min'] / kpis['num_reparaciones'],
    0  # Si no hay reparaciones, MTTR = 0
)

print("=== KPIs por Activo (primeros 10) ===")
print(kpis.head(10))

In [ ]:
# Guardar KPIs en formato Parquet - Capa Silver
kpis_s3_key = f"{SILVER_PREFIX}/kpis_activos.parquet"

# Guardar localmente primero y luego subir a S3
import tempfile
temp_file = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
kpis.to_parquet(temp_file.name, index=False)

# Subir a S3
s3_client.upload_file(temp_file.name, BUCKET_NAME, kpis_s3_key)
print(f"✓ KPIs guardados en: s3://{BUCKET_NAME}/{kpis_s3_key}")

# Limpieza
Path(temp_file.name).unlink()

## 9. Paso 6: Enriquecer con Datos de Activos y Generar Capa Silver

In [ ]:
# Combinar KPIs con información de activos
kpis_silver = kpis.merge(
    activos[['activo_id', 'nombre', 'tipo', 'ubicacion', 'fecha_puesta_servicio']],
    on='activo_id',
    how='left'
)

print("=== KPIs Silver (primeros 10) ===")
print(kpis_silver.head(10))
print(f"\nColumnas del dataset Silver: {kpis_silver.columns.tolist()}")

In [ ]:
# Guardar dataset enriquecido en Silver
kpis_silver_s3_key = f"{SILVER_PREFIX}/kpis_activos_silver.parquet"

# Guardar localmente y subir a S3
temp_file = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
kpis_silver.to_parquet(temp_file.name, index=False)
s3_client.upload_file(temp_file.name, BUCKET_NAME, kpis_silver_s3_key)
print(f"✓ Dataset Silver guardado en: s3://{BUCKET_NAME}/{kpis_silver_s3_key}")

# Limpieza
Path(temp_file.name).unlink()

## 10. Paso 7: Generar Capa Gold - KPIs de Negocio

### 7.1 KPIs por Tipo de Activo

In [ ]:
# KPIs por tipo de activo
kpis_por_tipo = kpis_silver.groupby('tipo').agg(
    num_activos=('activo_id', 'count'),
    disp_promedio=('disponibilidad_pct', 'mean'),
    mtbf_promedio=('mtbf_min', 'mean'),
    mttr_promedio=('mttr_min', 'mean')
).reset_index()

print("=== KPIs por Tipo de Activo ===")
print(kpis_por_tipo)

# Guardar en Gold
kpis_tipo_s3_key = f"{GOLD_PREFIX}/kpis_por_tipo.parquet"
temp_file = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
kpis_por_tipo.to_parquet(temp_file.name, index=False)
s3_client.upload_file(temp_file.name, BUCKET_NAME, kpis_tipo_s3_key)
print(f"\n✓ KPIs por tipo guardados en: s3://{BUCKET_NAME}/{kpis_tipo_s3_key}")
Path(temp_file.name).unlink()

### 7.2 KPIs por Ubicación

In [ ]:
# KPIs por ubicación
kpis_por_ubicacion = kpis_silver.groupby('ubicacion').agg(
    num_activos=('activo_id', 'count'),
    disp_promedio=('disponibilidad_pct', 'mean'),
    mtbf_promedio=('mtbf_min', 'mean'),
    mttr_promedio=('mttr_min', 'mean')
).reset_index()

print("=== KPIs por Ubicación ===")
print(kpis_por_ubicacion)

# Guardar en Gold
kpis_ubicacion_s3_key = f"{GOLD_PREFIX}/kpis_por_ubicacion.parquet"
temp_file = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
kpis_por_ubicacion.to_parquet(temp_file.name, index=False)
s3_client.upload_file(temp_file.name, BUCKET_NAME, kpis_ubicacion_s3_key)
print(f"\n✓ KPIs por ubicación guardados en: s3://{BUCKET_NAME}/{kpis_ubicacion_s3_key}")
Path(temp_file.name).unlink()

## 11. Resumen de Archivos Generados

In [ ]:
print("=== Archivos Generados en S3 ===")
print(f"\nCapa Silver:")
print(f"  - s3://{BUCKET_NAME}/{SILVER_PREFIX}/kpis_activos.parquet")
print(f"  - s3://{BUCKET_NAME}/{SILVER_PREFIX}/kpis_activos_silver.parquet")
print(f"\nCapa Gold:")
print(f"  - s3://{BUCKET_NAME}/{GOLD_PREFIX}/kpis_por_tipo.parquet")
print(f"  - s3://{BUCKET_NAME}/{GOLD_PREFIX}/kpis_por_ubicacion.parquet")

## 12. Modelo Dimensional Star Schema (Ejemplo: Dominio Mantenimiento)

A continuación, creamos un modelo star schema para el **dominio de Mantenimiento** como ejemplo de modelado dimensional. Este modelo permite análisis analítico eficiente de las órdenes de mantenimiento.

### Estructura del Modelo

```
                    ┌─────────────────┐
                    │  dim_fecha       │
                    │  ─────────────   │
                    │  fecha_id (PK)   │
                    │  fecha           │
                    │  anio            │
                    │  mes             │
                    │  dia             │
                    │  dia_semana      │
                    │  semana_anio     │
                    └────────┬────────┘
                             │
                             │
┌─────────────────┐  ┌──────┴───────┐  ┌─────────────────┐
│  dim_tipo_mant  │  │              │  │  dim_activo     │
│  ─────────────  │  │  fact_orden  │  │  ─────────────  │
│  tipo_mant_id   │  │  ──────────  │  │  activo_id (FK)│
│  tipo_mantenimiento│ │  orden_id  │  │  nombre         │
│  descripcion     │◄─┤  activo_id  │  │  tipo           │
│  categoria       │  │  fecha_id   │  │  ubicacion      │
└─────────────────┘  │  tipo_mant_id│  └─────────────────┘
                      │  horas_trabajo│
                      │  estado       │
                      └──────────────┘
```

### 12.1 Dimensión: dim_fecha

In [ ]:
# Crear dim_fecha a partir del rango de fechas en órdenes
# Convertir fechas de inicio y fin a datetime
ordenes['fecha_inicio_dt'] = pd.to_datetime(ordenes['fecha_inicio'])
ordenes['fecha_fin_dt'] = pd.to_datetime(ordenes['fecha_fin'])

# Generar rango de fechas único
fechas_unicas = pd.date_range(
    start=ordenes['fecha_inicio_dt'].min(),
    end=ordenes['fecha_fin_dt'].max(),
    freq='D'
)

# Crear dim_fecha
dim_fecha = pd.DataFrame({'fecha': fechas_unicas})
dim_fecha['fecha_id'] = dim_fecha['fecha'].dt.strftime('%Y%m%d').astype(int)
dim_fecha['anio'] = dim_fecha['fecha'].dt.year
dim_fecha['mes'] = dim_fecha['fecha'].dt.month
dim_fecha['dia'] = dim_fecha['fecha'].dt.day
dim_fecha['dia_semana'] = dim_fecha['fecha'].dt.dayofweek
dim_fecha['nombre_dia_semana'] = dim_fecha['fecha'].dt.day_name()
dim_fecha['semana_anio'] = dim_fecha['fecha'].dt.isocalendar().week

print(f"=== dim_fecha: {len(dim_fecha)} registros ===")
print(dim_fecha.head(10))

### 12.2 Dimensión: dim_tipo_mantenimiento

In [ ]:
# Crear dim_tipo_mantenimiento desde tipos únicos en órdenes
tipos_unicos = ordenes['tipo_mantenimiento'].dropna().unique()

dim_tipo_mant = pd.DataFrame({
    'tipo_mantenimiento': tipos_unicos
})
dim_tipo_mant['tipo_mant_id'] = range(1, len(dim_tipo_mant) + 1)

# Añadir categoría (agrupación de tipos)
categoria_map = {
    'PREVENTIVO': 'Planificado',
    'CORRECTIVO': 'No Planificado',
    'PREDICTIVO': 'Planificado'
}
dim_tipo_mant['categoria'] = dim_tipo_mant['tipo_mantenimiento'].map(categoria_map)

# Reordenar columnas
dim_tipo_mant = dim_tipo_mant[['tipo_mant_id', 'tipo_mantenimiento', 'categoria']]

print(f"=== dim_tipo_mantenimiento: {len(dim_tipo_mant)} registros ===")
print(dim_tipo_mant)

### 12.3 Dimensión: dim_activo

In [ ]:
# Crear dim_activo (dimensión de activos del modelo)
dim_activo = activos[['activo_id', 'nombre', 'tipo', 'ubicacion']].copy()

print(f"=== dim_activo: {len(dim_activo)} registros ===")
print(dim_activo.head())

### 12.4 Fact Table: fact_orden_mantenimiento

In [ ]:
# Crear fact_orden_mantenimiento
# Unir con dim_fecha para obtener fecha_id de inicio
ordenes_con_fecha = ordenes.copy()
ordenes_con_fecha['fecha_id'] = ordenes_con_fecha['fecha_inicio_dt'].dt.strftime('%Y%m%d').astype(int)

# Unir con dim_tipo_mant para obtener tipo_mant_id
ordenes_con_fecha = ordenes_con_fecha.merge(
    dim_tipo_mant[['tipo_mant_id', 'tipo_mantenimiento']],
    on='tipo_mantenimiento',
    how='left'
)

# Crear fact table con las claves foráneas
fact_orden = ordenes_con_fecha[['id_orden', 'activo_id', 'fecha_id', 'tipo_mant_id', 'horas_trabajo', 'estado']].copy()
fact_orden = fact_orden.rename(columns={'id_orden': 'orden_id'})

# Calcular duración en horas si tenemos fecha_fin
ordenes_con_fecha['duracion_horas'] = (
    (ordenes_con_fecha['fecha_fin_dt'] - ordenes_con_fecha['fecha_inicio_dt'])
    .dt.total_seconds() / 3600
)
fact_orden['duracion_horas'] = ordenes_con_fecha['duracion_horas']

print(f"=== fact_orden_mantenimiento: {len(fact_orden)} registros ===")
print(fact_orden.head(10))

### 12.5 Guardar Modelo Dimensional en S3

In [ ]:
# Guardar dimensiones y fact table en S3
DIM_PREFIX = "gold/dimensional"

# Guardar dim_fecha
temp_file = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
dim_fecha.to_parquet(temp_file.name, index=False)
s3_client.upload_file(temp_file.name, BUCKET_NAME, f"{DIM_PREFIX}/dim_fecha.parquet")
print(f"✓ dim_fecha → s3://{BUCKET_NAME}/{DIM_PREFIX}/dim_fecha.parquet")
Path(temp_file.name).unlink()

# Guardar dim_tipo_mantenimiento
temp_file = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
dim_tipo_mant.to_parquet(temp_file.name, index=False)
s3_client.upload_file(temp_file.name, BUCKET_NAME, f"{DIM_PREFIX}/dim_tipo_mantenimiento.parquet")
print(f"✓ dim_tipo_mantenimiento → s3://{BUCKET_NAME}/{DIM_PREFIX}/dim_tipo_mantenimiento.parquet")
Path(temp_file.name).unlink()

# Guardar dim_activo
temp_file = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
dim_activo.to_parquet(temp_file.name, index=False)
s3_client.upload_file(temp_file.name, BUCKET_NAME, f"{DIM_PREFIX}/dim_activo.parquet")
print(f"✓ dim_activo → s3://{BUCKET_NAME}/{DIM_PREFIX}/dim_activo.parquet")
Path(temp_file.name).unlink()

# Guardar fact_orden_mantenimiento
temp_file = tempfile.NamedTemporaryFile(suffix='.parquet', delete=False)
fact_orden.to_parquet(temp_file.name, index=False)
s3_client.upload_file(temp_file.name, BUCKET_NAME, f"{DIM_PREFIX}/fact_orden_mantenimiento.parquet")
print(f"✓ fact_orden_mantenimiento → s3://{BUCKET_NAME}/{DIM_PREFIX}/fact_orden_mantenimiento.parquet")
Path(temp_file.name).unlink()

### 12.6 Consultas Analíticas sobre el Modelo Star

Una vez tengamos el modelo star, podemos hacer consultas analíticas eficientes:

**1. Total de horas de mantenimiento por tipo y año:**

```python
# Join fact con dims
query = fact_orden.merge(dim_tipo_mant, on='tipo_mant_id') \
           .merge(dim_fecha, on='fecha_id') \
           .groupby(['anio', 'tipo_mantenimiento']) \
           .agg({'horas_trabajo': 'sum'}) \
           .reset_index()
```

**2. Órdenes por categoría (Planificado vs No Planificado):**

```python
query = fact_orden.merge(dim_tipo_mant, on='tipo_mant_id') \
           .groupby('categoria') \
           .agg({'orden_id': 'count', 'horas_trabajo': 'sum'}) \
           .reset_index()
```

**3. Top 5 activos con más horas de mantenimiento correctivo:**

```python
query = fact_orden.merge(dim_tipo_mant, on='tipo_mant_id') \
           .merge(dim_activo, on='activo_id') \
           .query("categoria == 'No Planificado'") \
           .groupby(['activo_id', 'nombre']) \
           .agg({'horas_trabajo': 'sum'}) \
           .sort_values('horas_trabajo', ascending=False) \
           .head(5)
```

### Resumen: Modelo Dimensional Creado

| Tabla | Tipo | Descripción |
|-------|------|-------------|
| `dim_fecha` | Dimensión | Calendario con niveles temporales |
| `dim_tipo_mantenimiento` | Dimensión | Tipos y categorías de mantenimiento |
| `dim_activo` | Dimensión | Catálogo de activos de TransCore |
| `fact_orden_mantenimiento` | Fact | Órdenes con claves foráneas a dims |

**Ubicación en S3:** `s3://{BUCKET_NAME}/gold/dimensional/`